# Play 1 — Agent Instructions Generator

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This utility notebook auto-generates the agent instructions for the Azure AI Foundry agent
by reading field metadata directly from a Tableau published datasource via the VizQL Data
Service API.

**No Fabric lakehouse required.** This notebook is self-contained — all it needs is a
Tableau PAT and a datasource LUID.

**How to use:**
1. Fill in Cell 1 with your Tableau credentials and datasource LUID
2. Run all cells
3. Copy the output from Cell 4
4. Paste it into the **Instructions** field when creating your Azure AI Foundry agent

**Finding your datasource LUID:**
- Tableau Cloud UI: open the datasource → the LUID is in the URL
- Tableau REST API: `GET /api/3.24/sites/{siteId}/datasources` → each datasource has an `id` field
- Play 2 (if run): query `Metadata_Lakehouse.dbo.tableau_datasources` → `datasource_id` column


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `PAT_NAME` | Tableau PAT name | Tableau → Account Settings → Personal Access Tokens |
| `PAT_SECRET` | Tableau PAT secret | Generated when you created the PAT |
| `POD` | Tableau Cloud pod hostname | First part of your Tableau Cloud URL |
| `SITE` | Site contentUrl slug | Your site URL slug. Use `""` for default site |
| `DATASOURCE_LUID` | LUID of the target datasource | See above |
| `DATASOURCE_NAME` | Display name for the datasource | Used in the generated instructions |

> **Note:** This notebook takes the PAT secret directly as a variable for simplicity.
> For production use, store the secret in Azure Key Vault and retrieve it via
> `notebookutils.credentials.getSecret()` as in the other Play notebooks.


## Cell 1 — Configuration

Set your Tableau credentials and datasource LUID here.

In [ ]:
# ── TABLEAU CONNECTION ────────────────────────────────────────────────────────
PAT_NAME         = ""   # PAT name from Tableau account settings
PAT_SECRET       = ""   # PAT secret value
POD              = ""   # e.g. 10ay.online.tableau.com
SITE             = ""   # Site contentUrl slug. Use "" for default site

# ── DATASOURCE ───────────────────────────────────────────────────────────────
DATASOURCE_LUID  = ""   # LUID of the published datasource to generate instructions for
DATASOURCE_NAME  = ""   # Display name e.g. "Superstore Datasource"

BASE = f"https://{POD}"

print("✓ Configuration loaded")
print(f"  Pod:              {POD}")
print(f"  Site:             {SITE or '(default)'}")
print(f"  Datasource LUID:  {DATASOURCE_LUID}")
print(f"  Datasource name:  {DATASOURCE_NAME}")


## Cell 2 — Authenticate to Tableau

In [ ]:
import requests

auth_response = requests.post(
    f"{BASE}/api/3.24/auth/signin",
    json={
        "credentials": {
            "personalAccessTokenName": PAT_NAME,
            "personalAccessTokenSecret": PAT_SECRET,
            "site": {"contentUrl": SITE}
        }
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"}
)
auth_response.raise_for_status()

auth_data = auth_response.json()
TOKEN   = auth_data["credentials"]["token"]
SITE_ID = auth_data["credentials"]["site"]["id"]

HEADERS = {
    "X-Tableau-Auth": TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print("✓ Authenticated to Tableau")
print(f"  Token:    {TOKEN[:8]}...")
print(f"  Site ID:  {SITE_ID}")


## Cell 3 — Fetch Field Metadata from VDS

In [ ]:
# Fetch field metadata from VDS
resp = requests.post(
    f"{BASE}/api/v1/vizql-data-service/read-metadata",
    json={"datasource": {"datasourceLuid": DATASOURCE_LUID}},
    headers=HEADERS
)
resp.raise_for_status()
all_fields = resp.json().get("data", [])

# Separate dimensions and measures — skip bins, sets, groups, calculated fields
dimensions = []
measures   = []

for f in all_fields:
    if f.get("columnClass") != "COLUMN":
        continue
    caption = f.get("fieldCaption", "")
    role    = f.get("fieldRole", "")
    if not caption:
        continue
    if role == "MEASURE":
        measures.append(caption)
    else:
        dimensions.append(caption)

print(f"✓ Fields retrieved from VDS")
print(f"  Dimensions: {len(dimensions)}")
print(f"  Measures:   {len(measures)}")
print(f"\n  Dimensions: {dimensions}")
print(f"  Measures:   {measures}")


## Cell 4 — Generate Agent Instructions

Run this cell and copy everything between the dividers into your Foundry agent Instructions field.

In [ ]:
# Format the field list for agent instructions
dim_list     = "\n".join([f"- {d}" for d in dimensions])
measure_list = "\n".join([f"- {m} (use SUM, AVG, MIN, MAX, or MEDIAN)" for m in measures])

instructions = f"""# Foundry Agent Instructions — {DATASOURCE_NAME}

> Paste this into the **Instructions** field when creating the Azure AI Foundry agent.

---

You are a data analyst agent with direct access to the {DATASOURCE_NAME} Tableau data source
via the VizQL Data Service API. You can query live Tableau data to answer business questions
in natural language.

You have access to the following fields:

**Dimensions** (use as-is, no aggregation needed):
{dim_list}

**Measures** (always aggregate — never request raw):
{measure_list}

When a user asks a data question:
1. Call queryTableauData with a well-constructed query_fields array containing only the
   fields needed to answer the question
2. Synthesize the returned data into a clear, concise natural language answer
3. Include specific numbers and rankings where relevant

QUERY CONSTRUCTION RULES:
- Always aggregate measures — never request a measure field without a function (SUM, AVG, MIN, MAX, MEDIAN)
- Always apply a date function to date fields — never request them as raw dates.
  Use YEAR for annual analysis, QUARTER or MONTH for trend analysis
- To count unique records, use a dimension field with function COUNTD
- Dimensions do not need a function
- Only request fields necessary to answer the question — keep payloads small
- Avoid high cardinality dimensions as standalone fields — they will produce oversized results

If the user asks something that can't be answered from this dataset, say so clearly.

---

## Notes for replication

- Authentication is handled internally by the Logic App — do not add a separate auth tool
- The queryTableauData tool is defined in the OpenAPI spec (openapi_spec.json)
- This field list was auto-generated from VDS read-metadata on {DATASOURCE_NAME}
"""

print("=" * 60)
print("COPY EVERYTHING BELOW THIS LINE INTO FOUNDRY AGENT INSTRUCTIONS")
print("=" * 60)
print()
print(instructions)
print("=" * 60)
print("END OF AGENT INSTRUCTIONS")
print("=" * 60)
